# 02 — Survival Analysis

Kaplan-Meier curves and Cox proportional hazards results for time-to-discontinuation (TTD).

In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

ttd = pd.read_csv('../../outputs/tables/ttd_events.csv')
cohort = pd.read_csv('../../outputs/tables/cohort_matched.csv')

if 'drug_class' not in ttd.columns:
    ttd = ttd.merge(cohort[['person_id', 'drug_class']], on='person_id', how='left')

ttd = ttd.dropna(subset=['ttd_days', 'discontinued'])
print(f'TTD events: {len(ttd):,}')
print(ttd.groupby('drug_class')[['ttd_days', 'discontinued']].agg(['median', 'mean', 'sum']).round(1))

TTD events: 5,000
           ttd_days                 discontinued           
             median   mean      sum       median mean   sum
drug_class                                                 
glp1          255.0  354.3   374157          1.0  1.0  1008
metformin     392.0  488.4  1452027          1.0  0.9  2752
sglt2         310.0  408.8   396948          1.0  0.9   910


In [2]:
# KM curves
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'metformin': '#3498DB', 'glp1': '#E74C3C', 'sglt2': '#2ECC71'}
kmfs = {}

for dc, color in colors.items():
    sub = ttd[ttd['drug_class'] == dc]
    if len(sub) == 0:
        continue
    kmf = KaplanMeierFitter()
    kmf.fit(sub['ttd_days'], event_observed=sub['discontinued'], label=dc)
    kmf.plot_survival_function(ax=ax, color=color, ci_show=True)
    kmfs[dc] = kmf

ax.set_xlabel('Days from index prescription')
ax.set_ylabel('Persistence probability')
ax.set_title('Kaplan-Meier: Treatment Persistence by Drug Class')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('../../outputs/figures/nb02_km_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved nb02_km_curves.png')

Saved nb02_km_curves.png


/var/folders/5_/fn5m9ggx60b349bz6bzp1qcc0000gn/T/ipykernel_55905/1841800718.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Log-rank test (metformin vs GLP-1)
met = ttd[ttd['drug_class'] == 'metformin']
glp = ttd[ttd['drug_class'] == 'glp1']
if len(met) > 0 and len(glp) > 0:
    lr = logrank_test(met['ttd_days'], glp['ttd_days'], event_observed_A=met['discontinued'], event_observed_B=glp['discontinued'])
    print(f'Log-rank (metformin vs GLP-1): p={lr.p_value:.4f}, test_stat={lr.test_statistic:.3f}')

Log-rank (metformin vs GLP-1): p=0.0000, test_stat=138.656


In [4]:
# Cox results
cox = pd.read_csv('../../outputs/tables/cox_ttd_results.csv')
print('Cox TTD hazard ratios:')
print(cox.to_string(index=False))

Cox TTD hazard ratios:
          covariate      coef  exp(coef)  se(coef)  coef lower 95%  coef upper 95%  exp(coef) lower 95%  exp(coef) upper 95%  cmp to         z            p  -log2(p)
     drug_class_num  0.134487   1.143950  0.017142        0.100889        0.168085             1.106154             1.183038     0.0  7.845355 4.317294e-15 47.718794
       age_at_index -0.000147   0.999853  0.001035       -0.002175        0.001882             0.997827             1.001884     0.0 -0.141554 8.874321e-01  0.172291
                cci  0.006340   1.006360  0.031857       -0.056098        0.068778             0.945446             1.071199     0.0  0.199014 8.422516e-01  0.247677
       hypertension  0.021560   1.021794  0.028019       -0.033358        0.076477             0.967193             1.079477     0.0  0.769449 4.416270e-01  1.179100
            obesity -0.038851   0.961894  0.030015       -0.097679        0.019977             0.906940             1.020178     0.0 -1.294391 1.95